## ResNet50

In [ ]:
import os   ### For working of CUDA #

# Disable XLA completely
os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=0"
os.environ["XLA_FLAGS"] = "--xla_gpu_cuda_data_dir=/usr/local/cuda"
os.environ["TF_DISABLE_XLA"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

In [ ]:

import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow.keras import layers, Model
from tensorflow.keras.applications.resnet import ResNet50, preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, accuracy_score, precision_score, recall_score, f1_score

# Constants - change if you want
IMG_SIZE = 224
BATCH_SIZE = 16            # lower if OOM; increase if GPU allows
AUTOTUNE = tf.data.AUTOTUNE
SEED = 1


## Loading Preprocessed Data

In [ ]:

df = pd.read_csv("../data/processed/manifest.csv")
df.head()

# Ensure columns: image_path,label,video_id,split
print(df.columns.tolist())

# Quick check: videos appearing in more than one split
dup = df.groupby("video_id")["split"].nunique()
leak_videos = dup[dup > 1]
print("Videos present in >1 split (should be zero):", len(leak_videos))
if len(leak_videos):
    print(leak_videos.head())


In [ ]:

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df   = df[df["split"] == "val"].reset_index(drop=True)
test_df  = df[df["split"] == "test"].reset_index(drop=True)

print("Counts (frames) -> train/val/test:", len(train_df), len(val_df), len(test_df))
print("Class balance in train:\n", train_df['label'].value_counts())


# Preprocessing

In [ ]:
# NEW WORKING DATA PIPELINE (NO py_function!)

def load_and_preprocess(path, label):
    # path: string tensor
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = preprocess_input(img)
    return img, label

def augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.05)
    img = tf.image.random_contrast(img, 0.9, 1.1)
    return img, label

# TRAIN DS
train_ds = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
train_ds = train_ds.shuffle(len(train_paths), seed=SEED)
train_ds = train_ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
train_ds = train_ds.map(augment, num_parallel_calls=AUTOTUNE)
train_ds = train_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

# VAL DS
val_ds = tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
val_ds = val_ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

# TEST DS
test_ds = tf.data.Dataset.from_tensor_slices((test_paths, test_labels))
test_ds = test_ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)


## Compute Class weight (imbalance)

In [ ]:
# CELL 5 - class weights
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(train_labels)
class_weights = compute_class_weight('balanced', classes=classes, y=train_labels)
class_weights = {int(c): float(w) for c,w in zip(classes, class_weights)}
print("Class weights:", class_weights)


## RESNET50 feature extractor head

In [ ]:
# CELL 6 - build model
base = ResNet50(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
base.trainable = False   # freeze

x = GlobalAveragePooling2D()(base.output)
x = Dropout(0.4)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.25)(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base.input, outputs=output)
model.compile(optimizer=Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


## Train head on frozen base

In [ ]:
# CELL 7 - callbacks and stage 1 fit
checkpoint_path = "resnet_stage1_best.h5"
callbacks = [
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1),
    ModelCheckpoint(checkpoint_path, monitor='val_loss', save_best_only=True, verbose=1)
]

history_stage1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weights,
    callbacks=callbacks
)


## Stage one curves

In [ ]:
# CELL 8 - plot stage1
def plot_history(h, title="History"):
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    plt.plot(h.history['loss'], label='train_loss')
    plt.plot(h.history['val_loss'], label='val_loss')
    plt.legend(); plt.title(title + " loss")
    plt.subplot(1,2,2)
    plt.plot(h.history['accuracy'], label='train_acc')
    plt.plot(h.history['val_accuracy'], label='val_acc')
    plt.legend(); plt.title(title + " accuracy")
    plt.show()

plot_history(history_stage1, "Stage 1")


## Fine Tuning (unfreeze last layers)

In [ ]:
# CELL 9 - fine-tuning
# Unfreeze last N layers
N = 50   # try 30 or 50; adjust if OOM
for layer in base.layers[:-N]:
    layer.trainable = False
for layer in base.layers[-N:]:
    layer.trainable = True

# Recompile with lower LR
model.compile(optimizer=Adam(1e-5), loss='binary_crossentropy', metrics=['accuracy'])

checkpoint_path_ft = "resnet_finetuned_best.h5"
callbacks_ft = [
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1),
    ModelCheckpoint(checkpoint_path_ft, monitor='val_loss', save_best_only=True, verbose=1)
]

history_stage2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weights,
    callbacks=callbacks_ft
)


## Stage 2 plot

In [ ]:
# CELL 10 - plot stage2 
plot_history(history_stage2, "Stage 2 (fine-tune)")


## Evaluate on test

In [ ]:
# CELL 11 - predict on val & test
# predict val probabilities for threshold selection
y_val_prob = model.predict(val_ds).ravel()
# collate y_val true in same order as val_ds
val_y_true = np.concatenate([y.numpy() for _, y in val_ds], axis=0)
print("val_y_true shape:", val_y_true.shape, "y_val_prob shape:", y_val_prob.shape)

# ROC & best threshold on VAL
fpr, tpr, thr = roc_curve(val_y_true, y_val_prob)
roc_auc_val = auc(fpr, tpr)
j_idx = np.argmax(tpr - fpr)
best_thr = thr[j_idx]
print("Val ROC AUC:", roc_auc_val, "Best threshold (Youden) from VAL:", best_thr)

# predict on test
y_test_prob = model.predict(test_ds).ravel()
test_y_true = np.concatenate([y.numpy() for _, y in test_ds], axis=0)
print("test shapes:", test_y_true.shape, y_test_prob.shape)

# get predictions using threshold from val
y_test_pred = (y_test_prob >= best_thr).astype(int)

print("Frame-level Test Metrics:")
print("Accuracy:", accuracy_score(test_y_true, y_test_pred))
print("Precision:", precision_score(test_y_true, y_test_pred, zero_division=0))
print("Recall:", recall_score(test_y_true, y_test_pred, zero_division=0))
print("F1:", f1_score(test_y_true, y_test_pred, zero_division=0))
print("\nClassification Report:\n", classification_report(test_y_true, y_test_pred, zero_division=0))


In [ ]:
# CELL 12 - confusion matrix and ROC for test
cm = confusion_matrix(test_y_true, y_test_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("ResNet50 — Test Confusion Matrix (frame-level)")
plt.show()

fpr_t, tpr_t, _ = roc_curve(test_y_true, y_test_prob)
roc_auc_test = auc(fpr_t, tpr_t)
plt.plot(fpr_t, tpr_t, label=f"AUC={roc_auc_test:.3f}")
plt.plot([0,1],[0,1],'k--')
plt.title("ResNet50 — Test ROC")
plt.legend()
plt.show()


In [ ]:
# CELL 14 - single image prediction helper
from tensorflow.keras.preprocessing import image

def predict_single_image(model, img_path, threshold=0.5, verbose=False):
    img = image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    x = preprocess_input(x)
    prob = float(model.predict(x).ravel()[0])
    pred = int(prob >= threshold)
    label = "FAKE" if pred == 1 else "REAL"
    if verbose:
        print(f"prob={prob:.4f}, pred={pred} -> {label}")
    return {"prob": prob, "pred": pred, "label": label}

# demo on 3 test images (replace with actual paths)
for p in list(test_df['image_path'].values[:3]):
    print(p, predict_single_image(model, p, threshold=best_thr, verbose=True))
